In [4]:
import torch
import torch.nn as nn
import numpy as np
import torch.optim as optim
import seisbench.data as sbd
import seisbench.generate as sbg 
from torch.utils.data import DataLoader, Subset
from predictor_pt_fixed import EQCCTModelS
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.utils.tensorboard import SummaryWriter

In [1]:
"""
Ensuring that the augmentations work properly on the training and validation datasets 
"""
def get_data_source2(): 
    data = sbd.STEAD(sampling_rate=150, component_order="ZNE")
    phase_dict = {
        "trace_s_arrival_sample": "S",
        "trace_S_arrival_sample": "S",
        "trace_S1_arrival_sample": "S",
        "trace_Sg_arrival_sample": "S",
        "trace_SmS_arrival_sample": "S",
        "trace_Sn_arrival_sample": "S"
    }

    augmentations = [sbg.WindowAroundSample(list(phase_dict.keys()), samples_before=2000, windowlen=6000, selection="first", strategy="pad", key=("X", "X")),
                    sbg.ProbabilisticLabeller(label_columns=phase_dict, model_labels='S', dim=0, sigma=3, shape='gaussian', key=("X", "y")), 
                    sbg.Normalize(demean_axis=-1, amp_norm_axis=-1, amp_norm_type="peak", key=("X", "X")),
                    sbg.ChangeDtype(np.float32, key=("X", "X"))]

    return data, augmentations 

def debug_shift(state_dict):
    _, md = state_dict["X"]
    print("→ After window, S-pick metadata:", md["trace_s_arrival_sample"])

# Load data 
device = "cuda" if torch.cuda.is_available() else "cpu"
data_source, augmentations = get_data_source2() 
train_data, val_data, _ = data_source.train_dev_test()

# Training Generator
train_generator = sbg.GenericGenerator(train_data)
train_generator.add_augmentations(augmentations)

import matplotlib.pyplot as plt

# Validation Generator
val_generator = sbg.GenericGenerator(val_data)
val_generator.add_augmentations(augmentations)
val_loader = DataLoader(val_generator, batch_size=16, shuffle=False)


def plot_batch(loader, n_samples=3, title="Batch"):
    batch = next(iter(loader))
    X = batch["X"].numpy()          # [B, 3, 6000]
    y = batch["y"].numpy()          # [B, C, 6000]

    # pick out the S‐channel (index 0)
    if y.ndim == 3:
        y = y[:, 0, :]              # now [B, 6000]

    for i in range(min(n_samples, X.shape[0])):
        fig, ax1 = plt.subplots(figsize=(12, 3))
        # plot waveform
        for ch in range(X.shape[1]):
            ax1.plot(X[i, ch] + ch*2, label=f"Ch {ch}")
        ax1.set_ylabel("Amplitude + offset")
        ax1.set_xlabel("Sample")
        ax1.set_title(f"{title} sample {i}")

        # overlay the true S‐pick Gaussian
        ax2 = ax1.twinx()
        ax2.plot(y[i], color="red", alpha=0.6, label="True S-label")
        ax2.set_ylabel("Label value")

        # combine legends
        lines, labs = ax1.get_legend_handles_labels()
        lines2, labs2 = ax2.get_legend_handles_labels()
        ax1.legend(lines + lines2, labs + labs2, loc="upper right")
        plt.show()

# After you define your loaders:
print("=== train_loader batch ===")
plot_batch(train_loader, n_samples=3, title="Train")
print("=== val_loader batch ===")
plot_batch(val_loader, n_samples=3, title="Val")



NameError: name 'torch' is not defined

In [2]:
"""
Find more example P/S waveform samples to validate the quality of the weight transfer from TF to PT 
"""
import seisbench.data as sbd

# Load dataset (with metadata)
txed = sbd.TXED(sampling_rate=100, component_order="ZNE")  # You already do this in main

# Metadata DataFrame
df = txed.metadata

# Phase columns to check (different pick names used in TXED)
p_pick_cols = [
    "trace_p_arrival_sample"
]

s_pick_cols = [
    "trace_s_arrival_sample"
]

# Find rows that have at least one non-null P and S pick
has_p = df[p_pick_cols].notna().any(axis=1)
has_s = df[s_pick_cols].notna().any(axis=1)

# Final filter
has_p_and_s = df[has_p & has_s]

# View a few sample indices
example_indices = has_p_and_s.index.tolist()
print(f"Found {len(example_indices)} examples with both P and S picks.")
print("Example indices:", example_indices[:10])


/home/skevofilaxc/miniconda3/envs/conveqcct/lib/python3.10/site-packages/seisbench/__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Found 312231 examples with both P and S picks.
Example indices: [207458, 207459, 207460, 207461, 207462, 207463, 207464, 207465, 207466, 207467]


In [3]:
from finetune_s_model import finetune_full_s_model


In [ ]:
finetune_full_s_model(smodel='ModelPS/modelS_STEAD_sigma20.pth', best_model='best_modelS_STEAD_sigma20_TXED_sigma10.pth', final_model='final_modelS_STEAD_sigma20_TXED_sigma10.pth')

Beginning Training Loop...
[Epoch 1] Avg Train Loss: 0.008052, Train F1 @0.40: 0.4849, Train Prec: 0.4188, Train Rec: 0.5758
[Epoch 1] Val Loss: 0.006800, Val F1 @0.40: 0.5627, Val Prec: 0.5387, Val Rec: 0.5890
  ↳ New best (F1: 0.5627) saved.
[Epoch 2] Avg Train Loss: 0.006775, Train F1 @0.40: 0.5485, Train Prec: 0.6575, Train Rec: 0.4705
[Epoch 2] Val Loss: 0.006158, Val F1 @0.40: 0.6185, Val Prec: 0.7179, Val Rec: 0.5433
  ↳ New best (F1: 0.6185) saved.
[Epoch 3] Avg Train Loss: 0.006453, Train F1 @0.40: 0.5668, Train Prec: 0.7362, Train Rec: 0.4608
[Epoch 3] Val Loss: 0.005925, Val F1 @0.40: 0.6346, Val Prec: 0.7524, Val Rec: 0.5487
  ↳ New best (F1: 0.6346) saved.
[Epoch 4] Avg Train Loss: 0.006238, Train F1 @0.40: 0.5841, Train Prec: 0.7594, Train Rec: 0.4746
[Epoch 4] Val Loss: 0.005766, Val F1 @0.40: 0.6471, Val Prec: 0.7685, Val Rec: 0.5588
  ↳ New best (F1: 0.6471) saved.
[Epoch 5] Avg Train Loss: 0.006085, Train F1 @0.40: 0.5962, Train Prec: 0.7696, Train Rec: 0.4866
[Epoch 

TypeError: unsupported format string passed to NoneType.__format__

: 